# Gate pipeline

One notebook for the **dataset pipeline's Gate steps**: **raw log → channel mix → judge → draft the labelled sample → probe (rules vs judge, cutoff sweep) → pick the cutoff → full pass → recipe preview → decide**. Not training: `parameter_test.ipynb` is that.

- **Two judge methods, one switch (`BACKEND`).** `"local"`: the Kev model loaded in this process (no server; `judge_local.py`). `"server"`: a llama-server with a decision model (`judge_client.py`). Both feed the same `GateJudge`, the same journal and the same reports; the journals of the two never mix (`by` carries the method, the model and the precision).
- **Runs on the Windows + CUDA machine, in its own virtualenv `.venv-kev`** (`py -3.13 -m venv .venv-kev`, `.venv-kev\Scripts\pip install -r requirements-kev.txt`, then `pip install --no-deps git+https://github.com/jaredpalmer/kev.git@fe64b12`). The other stacks cannot share it: Kev needs torch < 2.9 and transformers 5. Start it from that venv: `.venv-kev\Scripts\activate`, then `jupyter lab` from the repo root (or register the kernel once: `python -m ipykernel install --user --name resonance-lab-kev`, and pick it).
- **Free the GPU first:** a training run or llama-server holding it leaves the model no room. Kev-9B needs ~19 GB in bf16 (no 4-bit / 8-bit path): the cell below warns when it will not fit; `jaredpalmer/kev-4b` is ~9 GB.
- **The data stays on this machine.** The raw log, the sample and the journal are gitignored; **commit this notebook with its outputs cleared** (`tests/test_notebooks.py` checks).
- **Run All is safe:** the long pass and the draft are behind switches that are off. Plan and decisions: `.memory/roadmap/local-judge-and-gate-notebook-plan-2026-10-04.md`.

In [ ]:
import os
import subprocess
import sys
import time

# the notebook lives in notebooks/; everything runs from the repo root
ROOT = os.path.abspath(os.getcwd())
while not os.path.isfile(os.path.join(ROOT, "gate_pipeline.py")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise RuntimeError("Open this notebook from inside the resonance-lab repo.")
    ROOT = parent
os.chdir(ROOT)
for path in (ROOT, os.path.join(ROOT, "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

import categorize
import categorizer
import config
import gate_eval
import gate_judge
import gate_pipeline
from dataset_recipe import line_key, read_categories
from judge_client import SystemOneClient
from taxonomy import load_taxonomy

print("repo:  ", ROOT)
print("kernel:", sys.executable)
try:
    import torch
except ImportError:
    torch = None
    print("torch is not installed in this kernel: only BACKEND = \"server\" will work (the local judge needs .venv-kev)")
FREE_GB = None
if torch is not None and torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    FREE_GB = free_bytes / 1e9
    print(f"GPU {torch.cuda.get_device_name(0)}: {FREE_GB:.1f} of {total_bytes / 1e9:.1f} GB free")
elif torch is not None:
    print("no CUDA GPU: the local judge would run on the CPU (correct, slow)")

## 1 · Parameters
`BACKEND`, `RUN`, `DTYPE`, `DEVICE` pick the judge; `CUTOFF` is the smallest margin (top probability minus the second) that counts as an answer (`config.JUDGE_CUTOFF` is a guess: the probe below chooses it). `USE_CHANNEL` gives the judge the chat channel of a line when the raw log has one. `RECIPE` is a file of `configs/datasets/`.
Switches: `REFRESH_DATA` re-fetches and validates the raw log; `DRAFT_SAMPLE` writes the draft of the labelled sample; `RUN_FULL_PASS` judges the whole log (long). All off by default.

In [ ]:
RAW = config.RAW_LOGS
SAMPLE = config.GATE1_SAMPLE          # the hand-corrected labelled sample (gitignored)

BACKEND = "local"                     # "local": Kev in this process (.venv-kev) | "server": a llama-server
RUN = config.JUDGE_LOCAL_RUN          # local: a Hub id[@revision] or a directory; "jaredpalmer/kev-4b@v1.0" for a 16 GB card
DTYPE = "bf16"                        # local: "bf16" (half the memory) | "fp32" (the exact path)
DEVICE = None                         # local: "cuda" | "cpu" | None = cuda when there is one
SERVER_URL = config.JUDGE_URL         # server: llama-server's base URL
SERVER_MODEL = None                   # server: the model to ask when it runs several (router mode)

CUTOFF = config.JUDGE_CUTOFF
USE_CHANNEL = False
TARGET_PRECISION = 0.9                # the cutoff cell suggests the lowest cutoff that reaches it on the sample
RECIPE = "balanced-v1"
SAMPLE_SIZE = 200
SEED = 42

REFRESH_DATA = False                  # True: Fetch Data + Validate Data first (only when the raw data changed)
DRAFT_SAMPLE = False                  # True: write the draft of the labelled sample (step 4)
RUN_FULL_PASS = False                 # True: judge the whole raw log (step 7; resumable)

assert BACKEND in ("local", "server"), BACKEND
if BACKEND == "local":
    warning = gate_pipeline.memory_warning(RUN, DTYPE, None if DEVICE == "cpu" else FREE_GB)
    print(f"[WARNING] {warning}" if warning else f"local judge: {RUN} ({DTYPE}) on {DEVICE or ('cuda' if FREE_GB is not None else 'cpu')}")
else:
    print(f"llama-server judge: {SERVER_URL}")

## 2 · Data (only when it changed)
Fetch Data → Validate Data, each through `run_pipeline.py --only` (needs `huggingface_hub` for the fetch: it comes with transformers). Then the distinct lines of the raw log and their **channel mix**: the unified file of the pinned dataset revision has no channel, per-channel files do (`USE_CHANNEL` needs them).

In [ ]:
import pipelines

if REFRESH_DATA:
    for stage in pipelines.stages(pipelines.DEFAULT)[:2]:  # Fetch Data, Validate Data
        print(f"\n[->] {stage.name}")
        subprocess.run([sys.executable, os.path.join(ROOT, "run_pipeline.py"), "--only", stage.name], check=True, cwd=ROOT)
else:
    print("data stages skipped (REFRESH_DATA = False)")

if not os.path.exists(RAW):
    raise FileNotFoundError(f"{RAW} not found: set REFRESH_DATA = True (or RESONANCE_RAW_LOGS to a raw log)")
stats = {}
LINES = list(categorize.distinct_lines(RAW, stats))  # (key, original, channel)
print(f"\n{len(LINES):,} distinct lines in {RAW} ({stats['damaged']} damaged rows)")
print(gate_pipeline.format_channel_report(gate_pipeline.channel_report(LINES)))

## 3 · The judge (loaded once)
`check_server()` loads the model in local mode (the slow part: the first load downloads the weights, ~18 GB for Kev-9B) and answers one trivial question. Then three lines as a smoke test; the first request is slower than the rest (warm-up), so the speed measured here is rough: the probe measures it better.

In [ ]:
OPTIONS = gate_judge.choice_options(load_taxonomy())
if BACKEND == "local":
    from judge_local import LocalKevClient  # needs torch + kev: .venv-kev

    client, KIND = LocalKevClient(RUN, device=DEVICE, dtype=DTYPE), "kev-local"
else:
    client, KIND = SystemOneClient(SERVER_URL, model=SERVER_MODEL), "systemone"

started = time.time()
MODEL_NAME = client.check_server()
print(f"{MODEL_NAME}: ready in {time.time() - started:.0f} s")
BY = gate_judge.judge_id(MODEL_NAME, gate_judge.INSTRUCTIONS, OPTIONS, kind=KIND)
JUDGE = gate_judge.GateJudge(client, OPTIONS, CUTOFF)
print("judge id:", BY)

started = time.time()
for _, text, channel in LINES[:3]:
    answer = JUDGE.answer(text, channel if USE_CHANNEL else None)
    print(f"  {text[:40]!r:44} -> {answer.choice if answer else 'no answer'}" + (f" (margin {answer.margin:.2f})" if answer else ""))
SECONDS_PER_LINE = (time.time() - started) / 3
print(f"~{SECONDS_PER_LINE:.2f} s per line")

## 4 · Draft the labelled sample
The judge only has to beat the rules **on real chat**, so the sample must be real and hand-labelled. This cell draws `SAMPLE_SIZE` lines stratified by the **rules'** guess (so rare categories are in), lets the **judge pre-label** them, and writes `gate1-sample.draft.jsonl` (with the judge's `margin`: the low ones are where to look first). **You correct the labels in an editor and save the result as `gate1-sample.jsonl`**: the judge pre-labels, you decide. A line the judge gave no answer for is labelled `?`, which the sample reader refuses until you replace it. Lines already in the sample are not drawn again; an existing draft is never overwritten.

In [ ]:
DRAFT = gate_pipeline.draft_path(SAMPLE)
if DRAFT_SAMPLE:
    known = {line_key(row["original"]) for row in gate_eval.read_sample(SAMPLE)} if os.path.exists(SAMPLE) else set()
    rows = gate_pipeline.draft_sample(LINES, SAMPLE_SIZE, SEED, guess=lambda text, channel: categorizer.categorize(text), exclude=known)
    for row in rows:
        answer = JUDGE.answer(row["original"], row.get("channel") if USE_CHANNEL else None)
        row["category"] = answer.choice if answer else gate_pipeline.UNGUESSED
        if answer:
            row["margin"] = round(answer.margin, 3)
    gate_pipeline.write_draft(DRAFT, rows)  # FileExistsError when a draft is already there: move it first
    print(f"{len(rows)} lines drafted -> {DRAFT}\nCorrect the `category` of each, delete the `margin` if you like, save as {SAMPLE}")
else:
    print("skipped (DRAFT_SAMPLE = False)")

## 5 · Probe: the rules and the judge on the labelled sample
Needs `gate1-sample.jsonl`. The judge is asked **once** per line (answers are cached); the cutoff sweep and both reports come from that pass. A sample of a few hundred lines **overfits the cutoff**, and each root has ~20 lines: read the counts, not only the percentages, and look at what each root was taken for.

In [ ]:
ROWS = None
if not os.path.exists(SAMPLE):
    print(f"no labelled sample at {SAMPLE}: draft it in step 4 and correct it by hand")
else:
    FULL_SAMPLE = gate_eval.read_sample(SAMPLE)
    PLAIN_SAMPLE = [{"original": row["original"], "category": row["category"]} for row in FULL_SAMPLE]
    ROWS = FULL_SAMPLE if USE_CHANNEL else PLAIN_SAMPLE

    def score(rows):
        channels = {row["original"]: row.get("channel") for row in rows}
        return gate_eval.evaluate_categorizer(rows, lambda text: JUDGE.predict(text, channels.get(text)))

    started = time.time()
    SWEEP = gate_judge.cutoff_sweep(ROWS, JUDGE, gate_judge.SWEEP_CUTOFFS)
    SECONDS_PER_LINE = (time.time() - started) / len(ROWS)
    RULES_REPORT = gate_eval.evaluate_categorizer(ROWS, categorizer.categorize)
    JUDGE_REPORT = score(ROWS)
    print(gate_eval.format_gate_report(RULES_REPORT, categorizer.BASELINE))
    print("\n" + gate_eval.format_gate_report(JUDGE_REPORT, f"{BY}, cutoff {CUTOFF}"))
    if JUDGE.failures:
        print(f"\n{JUDGE.failures} request(s) failed: counted as no answer.")
    print("\n" + gate_judge.format_sweep(SWEEP, BY))
    print(f"\n{SECONDS_PER_LINE:.2f} s per line on this pass")
    if any("channel" in row for row in FULL_SAMPLE):  # the sample has channels: with and without, same judge
        for label, rows in (("without channel", PLAIN_SAMPLE), ("with channel", FULL_SAMPLE)):
            report = score(rows)
            print(f"{label:16} accuracy {report['accuracy']:.0%}, coverage {report['coverage']:.0%}, precision {report['precision'] or 0:.0%}")

## 6 · Pick the cutoff
The suggestion is the lowest cutoff whose precision on the sample reaches `TARGET_PRECISION` (lines under the cutoff stay uncategorized: precision against coverage is the trade). Set `CUTOFF` above and re-run from step 3, or make it the default in `config.py` (`JUDGE_CUTOFF`). The same pass from a shell: `python scripts\categorize.py --judge-local RUN --dtype DTYPE --cutoff X` (or `--judge-url`).

In [ ]:
if ROWS:
    ok = [s for s in SWEEP if s["precision"] is not None and s["precision"] >= TARGET_PRECISION]
    if ok:
        best = ok[0]
        print(f"lowest cutoff with precision >= {TARGET_PRECISION:.0%}: {best['cutoff']} (coverage {best['coverage']:.0%}, {best['covered']} of {len(ROWS)} lines answered)")
    else:
        print(f"no cutoff reaches precision {TARGET_PRECISION:.0%} on this sample: the judge is not good enough for this question, or the sample is small")
    print(f"current CUTOFF = {CUTOFF}")

## 7 · Full pass over the raw log
Resumable: every answer is flushed to the journal (`data/processed/gate1_judge.jsonl`) as it comes, a run that stops (Ctrl+C, a crash) continues from there, and a journal of another judge (method, model, precision or question) stops the run instead of mixing. The categories file is **derived from the journal for the cutoff**, so a new cutoff needs no new request. Only `RUN_FULL_PASS = True` starts it.

In [ ]:
print(f"estimate for the full pass: {len(LINES):,} lines at {SECONDS_PER_LINE:.2f} s per line = about {len(LINES) * SECONDS_PER_LINE / 60:.0f} min "
      "(the speed of the probe, or of three lines: rough; lines already in the journal are not asked again)")
if RUN_FULL_PASS:
    try:
        categorize.judge_pass(RAW, config.JUDGE_PASS_FILE, config.CATEGORIES_FILE, JUDGE, BY, CUTOFF, USE_CHANNEL,
                              where=RUN if BACKEND == "local" else SERVER_URL, who="the model" if BACKEND == "local" else "the server")
    except categorize.PassError as error:
        print(f"[ERROR] {error}")
else:
    print("skipped (RUN_FULL_PASS = False)")

## 8 · Recipe preview
What the recipe would take from the categories: its share per category against the natural share of the log, the lines available and selected, and **which category limits the dataset**. An upper bound: `preprocess.py` drops more lines afterwards (duplicates, eval overlap, spam filters). Run the real thing with `python preprocess.py --recipe NAME`.

In [ ]:
if os.path.exists(config.CATEGORIES_FILE):
    CATEGORIES = read_categories(config.CATEGORIES_FILE)
    PREVIEW = gate_pipeline.recipe_preview_by_name(RECIPE, CATEGORIES, [key for key, _, _ in LINES])
    print(gate_pipeline.format_recipe_preview(PREVIEW, RECIPE))
else:
    print(f"no categories file yet ({config.CATEGORIES_FILE}): run step 7, or `python scripts\\categorize.py` for the rules")

## 9 · Decide / record
The block to paste into the memory notes (`.memory/roadmap/`): the judge, the cutoff, its scores against the rules on the sample, the speed and the full pass. Then free the GPU.

In [ ]:
if ROWS:
    categorized = len(read_categories(config.CATEGORIES_FILE)) if RUN_FULL_PASS and os.path.exists(config.CATEGORIES_FILE) else None
    print(gate_pipeline.decision_text(by=BY, cutoff=CUTOFF, judge=JUDGE_REPORT, rules=RULES_REPORT, seconds_per_line=round(SECONDS_PER_LINE, 2),
                                      lines=len(LINES) if categorized is not None else None, categorized=categorized))
else:
    print("nothing to record yet: needs the labelled sample (step 5)")

In [ ]:
# free the GPU for whatever runs next (a training, llama-server)
import gc

JUDGE = client = None  # drop the references: the model is freed with them
gc.collect()
if torch is not None and torch.cuda.is_available():
    torch.cuda.empty_cache()
print("judge released")